In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# names.txt is downloaded from https://github.com/karpathy/makemore/blob/master/names.txt
words = open('../data/makemore/names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

In [ ]:
block_size = 3  # context length

def build_dataset(words):
    X, Y = [], []
    for w in words:  # Or set the [:5] to look at the examples and check mini-batch overfitting
        # print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix]  # Shift the context window

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])      # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])      # 10%

In [ ]:
n_embd = 10
n_hidded = 200

# Defining the parameters
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidded), generator=g) * 0.2  # 2) Making the output closer to 0 to reduce tanh saturation (fixing the tanh)
b1 = torch.randn(n_hidded, generator=g) * 0.01  # 2) Making the output closer to 0 to reduce tanh saturation (fixing the tanh)
W2 = torch.randn((n_hidded, vocab_size), generator=g) * 0.01  # 1) Making the output layer weights more uniformly distributied (fixing the loss)
b2 = torch.randn(vocab_size, generator=g) * 0.0  # 1) Making the output layer weights more uniformly distributied (fixing the loss)

# The ratios during the initializaation is a dummy way to make the distributions for this NN more stable:
# 1) Uniform distribution of the output logits
# 2) Close to zero weights as an input for tanh to reduce its saturation
# But there is another problem: distribution tails becomes larger after matrix multiplication (the distribution is squizes - std increases)
# To mitigate this we will need to multiply the initial weights by a coefficient or use torch.nn.init special functions
# (e.g., they based on the Kaiming etc. paper).
# The new initialization of the W1 is presented bellow

W1 = torch.randn((n_embd * block_size, n_hidded), generator=g) * (5/3) / (n_embd * block_size ** 0.5)  # The Kaiming init (https://docs.pytorch.org/docs/2.9/nn.init.html)

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # Mini-batch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))  # Batch size is batch_size
    Xb, Yb = Xtr[ix], Ytr[ix]

    # Forward pass
    emb = C[Xb]  # Embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1)  # Concatenate the vectors
    hpreact = embcat @ W1 + b1  # Hidden layer pre-activation
    h = torch.tanh(hpreact)  # Hidden layer
    logits = h @ W2 + b2  # Output layer
    loss = F.cross_entropy(logits, Ytr[ix])

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Parameters update
    lr = 0.1 if i < 100000 else 0.01  # Step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # Track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    # break

print(f'{max_steps}/{max_steps}: {loss.item():.4f}')

Fixing the loss by making the output logits distribution more nuiform like

In [ ]:
# # The initial loss we want to expect
# # But initializing the weights with normal distribution we have a way higher value: 25.55
# # We decided to scale weights by 0.01 (* 0.01) and make biases equal to 0 (* 0.0)
# -torch.tensor(1/27.0).log()

Fixing the saturated tanh function

In [ ]:
# # tanh output distribution
# plt.hist(h.view(-1).tolist(), bins=50)

In [ ]:
# # tanh input distribution
# plt.hist(hpreact.view(-1).tolist(), bins=50)

In [ ]:
# # Looking at the binary map of the tanh activations for a minibatch after the initialization
# # Too many white pixels tells that the gradient will be vanished during the back propogation
# # (tanh values are too high: [-1; 1 - tanh is saturated]), because the gradient will be zero
# plt.figure(figsize=(20, 10))
# plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest')

In [ ]:
plt.plot(lossi)

In [ ]:
@torch.no_grad()  # Do not require the grad for the only forward pass (for efficiency)
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x]  # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)  # Concat into (N, block_size * n_embd)
    h = torch.tanh(embcat @ W1 + b1)  # (N, n_hidden)
    logits = h @ W2 + b2  # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# Sampling
inf_gen = torch.Generator().manual_seed(2147483647 + 10)
for i in range(20):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, 1)

        idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=inf_gen).item()
        context = context[1:] + [idx]
        out.append(itos[idx])
        if idx == 0:
            break

    print(''.join(out).strip('.'))